# Gold Price Analytics Project

**Name:** Ritesh Wasekar  
**Roll Number:** SA223

This notebook analyzes a real monthly gold price dataset. The goal is to understand long-term price behavior and build a simple model that predicts gold price from time-based features and monthly price movement.

## Problem Statement

I am analyzing a gold price dataset to understand how gold prices changed over time and what simple patterns are visible in the data. This can help investors, students, and businesses understand price trends and monthly fluctuations.

**Dataset used:** Monthly Gold Prices dataset  
**Source:** datasets/gold-prices repository on GitHub  
**Rows and columns after preparation:** 2319 rows and 8 columns  

**Column meanings:**
- `Date`: month of the record
- `Price`: gold price in USD per ounce
- `Year`: year extracted from the date
- `Month_Num`: numeric month
- `Month_Name`: month name
- `Quarter`: quarter of the year
- `Decade`: decade group such as 1980s or 2000s
- `Price_Change`: month-to-month change in price

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

The required libraries are imported in one place so the notebook can run from top to bottom without errors.

In [ ]:
df = pd.read_csv('gold_analysis_dataset.csv', parse_dates=['Date'])
df.head()

The first five rows confirm that the dataset loaded correctly and that the engineered columns are available for analysis.

In [ ]:
print('Shape:', df.shape)
print('\nData Types:\n', df.dtypes)

The dataset contains more than 1000 rows and has both numeric and categorical columns, which matches the assignment requirements.

In [ ]:
print('Missing values before cleaning:\n', df.isnull().sum())
print('Duplicate rows before cleaning:', df.duplicated().sum())

# The only potential missing value came from monthly difference calculation, and it has already been filled with 0.
df['Price_Change'] = df['Price_Change'].fillna(0)
df = df.drop_duplicates()

print('\nMissing values after cleaning:\n', df.isnull().sum())
print('Duplicate rows after cleaning:', df.duplicated().sum())

No missing values remained after cleaning. Duplicate rows were also removed, so the final dataset is ready for statistics and modeling.

In [ ]:
def calc_stats(series):
    return {
        'Mean': series.mean(),
        'Median': series.median(),
        'Mode': series.mode().iloc[0],
        'Standard Deviation': series.std(),
        'Variance': series.var(),
        'Range': series.max() - series.min(),
        'Mid-range': (series.max() + series.min()) / 2
    }

stats_df = pd.DataFrame({
    'Price': calc_stats(df['Price']),
    'Price_Change': calc_stats(df['Price_Change'])
}).round(2)
stats_df

The price column has a very large spread because the dataset covers a long historical period. The price change column is centered much closer to zero, which is expected because most month-to-month changes are smaller than the full price level.

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['Price'], bins=30, edgecolor='black')
plt.title('Histogram of Gold Price')
plt.xlabel('Price (USD per ounce)')
plt.ylabel('Frequency')
plt.show()

The histogram is right-skewed, which means low historical prices appear much more often than very high recent prices.

In [ ]:
plt.figure(figsize=(8,4))
sns.countplot(data=df, x='Quarter', order=['Q1','Q2','Q3','Q4'])
plt.title('Count of Records by Quarter')
plt.xlabel('Quarter')
plt.ylabel('Count')
plt.show()

The quarter count chart is balanced because the dataset contains monthly records over many years, so each quarter appears almost equally often.

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df['Price_Change'])
plt.title('Boxplot of Monthly Price Change')
plt.xlabel('Price Change (USD)')
plt.show()

The boxplot shows many outliers, which means some months had unusually large jumps or drops in gold price.

In [ ]:
plt.figure(figsize=(8,5))
corr = df[['Price','Year','Month_Num','Price_Change']].corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='YlGnBu', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

The heatmap shows a strong positive relationship between price and year, which means gold prices generally increased over the long run.

In [ ]:
X = df[['Year','Month_Num','Month_Name','Quarter','Decade','Price_Change']]
y = df['Price']

numeric_features = ['Year','Month_Num','Price_Change']
categorical_features = ['Month_Name','Quarter','Decade']

preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])

model = RandomForestRegressor(n_estimators=200, random_state=42)
pipe = Pipeline([('preprocess', preprocess), ('model', model)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print('R-squared:', round(r2_score(y_test, pred), 4))
print('MAE:', round(mean_absolute_error(y_test, pred), 2))
print('RMSE:', round((mean_squared_error(y_test, pred))**0.5, 2))

The model performs very well on this dataset because the year, decade, and monthly change already contain strong information about the target price. In simple English, the model can estimate gold price quite accurately from the time period and recent movement.

## Insights and Recommendations

### Findings
1. **Finding 1:** The histogram shows that the data is strongly right-skewed, so high gold prices are mainly concentrated in recent years.
2. **Finding 2:** The statistics table shows that the mean price is much higher than the median price, which confirms that recent high values pull the average upward.
3. **Finding 3:** The heatmap shows a strong positive correlation between `Price` and `Year`, meaning gold has generally trended upward over time.
4. **Finding 4:** The boxplot of `Price_Change` shows several outliers, so some months experienced unusually high volatility.

### Recommendations
1. For a non-technical reader: track long-term trend and avoid judging gold only by one or two monthly movements because short-term changes can be noisy.
2. For a non-technical reader: pay extra attention during highly volatile months because the boxplot suggests that sudden jumps and drops do happen.